# 📈 Customer Lifetime Value (LTV) Modelling

> **Dataset:** Online Retail (UCI Machine Learning Repository)  
> **Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
> **Goal:** Estimate the future value of each customer using a probabilistic model, segment customers by predicted LTV, and surface actionable recommendations for retention and revenue growth.

---

## Business Question

> *"How much revenue can we expect each customer to generate over the next 12 months — and which customers should we prioritise?"*

We answer this using the **BG/NBD + Gamma-Gamma** framework, the industry standard for non-contractual LTV modelling (used by Amazon, Spotify, and most growth-stage e-commerce companies).

### How it works
| Model | What it predicts |
|---|---|
| **BG/NBD** | How many purchases a customer will make in the future |
| **Gamma-Gamma** | How much each purchase will be worth on average |
| **LTV = BG/NBD × Gamma-Gamma** | Total expected revenue over a time horizon |

---
## 1. Setup & Imports

In [ ]:
# Install required libraries if needed
# !pip install lifetimes openpyxl matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
from lifetimes.plotting import (
    plot_frequency_recency_matrix,
    plot_probability_alive_matrix,
    plot_period_transactions
)

# ── Plotting style ──────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)

print('Libraries loaded ✅')

---
## 2. Load & Inspect the Data

Download the dataset from UCI: https://archive.ics.uci.edu/dataset/352/online+retail  
Save it as `online_retail.xlsx` inside a `data/` folder in this directory.

In [ ]:
df = pd.read_excel('data/online_retail.xlsx', dtype={'CustomerID': str})

print(f'Rows:             {len(df):,}')
print(f'Columns:          {list(df.columns)}')
print(f'Date range:       {df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}')
print(f'Unique customers: {df.CustomerID.nunique():,}')
print(f'Unique products:  {df.StockCode.nunique():,}')
df.head()

---
## 3. Data Cleaning

In [ ]:
raw_rows = len(df)

# 1. Remove cancellations
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# 2. Drop rows missing CustomerID
df = df.dropna(subset=['CustomerID'])

# 3. Remove negative or zero quantity/price
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 4. Remove obvious test/admin entries
df = df[~df['Description'].str.upper().str.contains('TEST|MANUAL|AMAZON|POSTAGE|BANK', na=False)]

# 5. Add revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 6. Restrict to UK (largest market, cleaner data)
df_uk = df[df['Country'] == 'United Kingdom'].copy()

print(f'Raw rows:         {raw_rows:,}')
print(f'Clean rows (all): {len(df):,}')
print(f'UK rows:          {len(df_uk):,}')
print(f'UK customers:     {df_uk.CustomerID.nunique():,}')
print(f'Removed:          {raw_rows - len(df_uk):,} rows ({(raw_rows - len(df_uk))/raw_rows:.1%})')

---
## 4. Build the RFM Summary Table

The BG/NBD model requires each customer to be summarised into **3 metrics**:

| Metric | Definition |
|---|---|
| **frequency** | Number of repeat purchases (total purchases minus 1) |
| **recency** | Time between first and last purchase (in weeks) |
| **T** | Total observation period — time since first purchase (in weeks) |
| **monetary_value** | Average revenue per transaction |

In [ ]:
# Use the last observed date as the analysis cutoff
cutoff_date = df_uk['InvoiceDate'].max()

rfm = summary_data_from_transaction_data(
    df_uk,
    customer_id_col='CustomerID',
    datetime_col='InvoiceDate',
    monetary_value_col='Revenue',
    observation_period_end=cutoff_date,
    freq='W'  # weekly granularity
)

# Keep only customers with at least 1 repeat purchase (required for Gamma-Gamma)
rfm_repeat = rfm[rfm['frequency'] > 0].copy()

print(f'Total customers:         {len(rfm):,}')
print(f'Repeat purchasers:       {len(rfm_repeat):,} ({len(rfm_repeat)/len(rfm):.1%})')
print(f'Avg frequency:           {rfm_repeat.frequency.mean():.1f} repeat purchases')
print(f'Avg monetary value:      £{rfm_repeat.monetary_value.mean():.2f} per transaction')
rfm_repeat.head(10)

---
## 5. Fit the BG/NBD Model

The **Beta-Geometric / Negative Binomial Distribution (BG/NBD)** model estimates:
- The probability that a customer is still active (not churned)
- How many purchases they will make in a future period

In [ ]:
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(
    rfm_repeat['frequency'],
    rfm_repeat['recency'],
    rfm_repeat['T']
)

print('BG/NBD model fitted ✅')
print(bgf.summary)

### 5a. Model Validation — Predicted vs Actual Transactions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_period_transactions(bgf, ax=ax)
ax.set_title('BG/NBD Model: Predicted vs Actual Weekly Transactions')
ax.set_xlabel('Number of Transactions')
ax.set_ylabel('Customers')
ax.legend(['Actual', 'Model'], framealpha=0.7)
plt.tight_layout()
plt.savefig('outputs/bgnbd_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

### 5b. Frequency × Recency Matrix

This matrix shows **expected future purchases** for a customer given their recency and frequency. High-recency + high-frequency customers in the top-right are the most valuable.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_frequency_recency_matrix(bgf, ax=ax, cmap='YlOrBr')
ax.set_title('Expected Future Purchases: Frequency × Recency Matrix')
plt.tight_layout()
plt.savefig('outputs/frequency_recency_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 5c. Probability Alive Matrix

For each frequency/recency combination, what is the probability the customer is still active?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_probability_alive_matrix(bgf, ax=ax, cmap='YlOrBr')
ax.set_title('Probability Customer is Still Active: Frequency × Recency Matrix')
plt.tight_layout()
plt.savefig('outputs/probability_alive_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Fit the Gamma-Gamma Model

The **Gamma-Gamma** model estimates the **average monetary value** of each customer's future transactions, assuming it's independent of purchase frequency.

In [ ]:
# Verify low correlation between frequency and monetary value (assumption check)
corr = rfm_repeat[['frequency', 'monetary_value']].corr().iloc[0, 1]
print(f'Correlation (frequency vs monetary value): {corr:.3f}')
if abs(corr) < 0.3:
    print('✅ Low correlation — Gamma-Gamma assumption holds')
else:
    print('⚠️  Higher than expected — interpret results with caution')

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    rfm_repeat['frequency'],
    rfm_repeat['monetary_value']
)

print('\nGamma-Gamma model fitted ✅')
print(ggf.summary)

---
## 7. Calculate 12-Month LTV

In [ ]:
# Predict LTV over next 12 months (52 weeks), assuming 10% annual discount rate
rfm_repeat['predicted_ltv_12m'] = ggf.customer_lifetime_value(
    bgf,
    rfm_repeat['frequency'],
    rfm_repeat['recency'],
    rfm_repeat['T'],
    rfm_repeat['monetary_value'],
    time=52,           # weeks
    freq='W',
    discount_rate=0.01 # weekly discount rate ≈ 10% annually
)

# Probability still alive
rfm_repeat['prob_alive'] = bgf.conditional_probability_alive(
    rfm_repeat['frequency'],
    rfm_repeat['recency'],
    rfm_repeat['T']
)

print(f'LTV summary (12-month horizon):')
print(rfm_repeat['predicted_ltv_12m'].describe().apply(lambda x: f'£{x:.2f}'))

---
## 8. Customer Segmentation by LTV

We segment customers into 4 tiers using quartiles of predicted LTV.

In [ ]:
rfm_repeat['ltv_segment'] = pd.qcut(
    rfm_repeat['predicted_ltv_12m'],
    q=4,
    labels=['Low', 'Mid', 'High', 'Champions']
)

segment_summary = rfm_repeat.groupby('ltv_segment', observed=True).agg(
    customers        = ('predicted_ltv_12m', 'count'),
    avg_ltv          = ('predicted_ltv_12m', 'mean'),
    total_ltv        = ('predicted_ltv_12m', 'sum'),
    avg_frequency    = ('frequency', 'mean'),
    avg_order_value  = ('monetary_value', 'mean'),
    avg_prob_alive   = ('prob_alive', 'mean')
).round(2)

segment_summary['revenue_share'] = (
    segment_summary['total_ltv'] / segment_summary['total_ltv'].sum()
).map('{:.1%}'.format)

segment_summary['avg_ltv'] = segment_summary['avg_ltv'].map('£{:.2f}'.format)
segment_summary['total_ltv'] = segment_summary['total_ltv'].map('£{:,.0f}'.format)
segment_summary['avg_order_value'] = segment_summary['avg_order_value'].map('£{:.2f}'.format)
segment_summary['avg_prob_alive'] = segment_summary['avg_prob_alive'].map('{:.1%}'.format)

segment_summary

---
## 9. Visualisations

### 9a. LTV Distribution by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = [SAND, CAMEL, BROWN, CHOCOLATE]
segment_order = ['Low', 'Mid', 'High', 'Champions']

# ── Box plot: LTV distribution per segment ──
ax = axes[0]
data_plot = [rfm_repeat[rfm_repeat['ltv_segment'] == s]['predicted_ltv_12m'].values
             for s in segment_order]
bp = ax.boxplot(data_plot, patch_artist=True, labels=segment_order,
                medianprops=dict(color=TERRACOTTA, linewidth=2))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
ax.set_title('LTV Distribution by Segment')
ax.set_xlabel('Segment')
ax.set_ylabel('Predicted 12-Month LTV (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.grid(axis='y')

# ── Bar chart: Revenue share per segment ──
ax2 = axes[1]
rev_data = rfm_repeat.groupby('ltv_segment', observed=True)['predicted_ltv_12m'].sum()
rev_data = rev_data.reindex(segment_order)
bars = ax2.bar(segment_order, rev_data.values, color=palette, edgecolor='white', linewidth=0.5)
total = rev_data.sum()
for bar, val in zip(bars, rev_data.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
             f'{val/total:.1%}', ha='center', va='bottom', fontsize=9, color=CHOCOLATE)
ax2.set_title('Total Predicted Revenue Share by Segment')
ax2.set_xlabel('Segment')
ax2.set_ylabel('Total Predicted LTV (£)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax2.grid(axis='y')

plt.suptitle('12-Month LTV Segmentation', fontsize=14, y=1.02, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/ltv_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

### 9b. LTV vs Probability Alive — Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

color_map = {'Low': SAND, 'Mid': CAMEL, 'High': BROWN, 'Champions': CHOCOLATE}

for seg in segment_order:
    subset = rfm_repeat[rfm_repeat['ltv_segment'] == seg]
    ax.scatter(
        subset['prob_alive'],
        subset['predicted_ltv_12m'],
        alpha=0.45,
        s=25,
        color=color_map[seg],
        label=seg,
        edgecolors='none'
    )

# Quadrant lines
ax.axvline(0.5, color=TERRACOTTA, linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(
    rfm_repeat['predicted_ltv_12m'].median(),
    color=TERRACOTTA, linestyle='--', linewidth=0.8, alpha=0.6
)

# Quadrant labels
ymax = rfm_repeat['predicted_ltv_12m'].quantile(0.97)
ax.text(0.02, ymax * 0.92, 'At-risk\nhigh-value', fontsize=8, color=TERRACOTTA, style='italic')
ax.text(0.75, ymax * 0.92, 'Champions\n(retain & upsell)', fontsize=8, color=TERRACOTTA, style='italic')
ax.text(0.02, ymax * 0.05, 'Dormant\nlow-value', fontsize=8, color=CAMEL, style='italic')
ax.text(0.75, ymax * 0.05, 'Loyal\nbut low spend', fontsize=8, color=CAMEL, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, ymax)
ax.set_xlabel('Probability Customer is Still Active')
ax.set_ylabel('Predicted 12-Month LTV (£)')
ax.set_title('Customer Map: LTV vs Probability Alive')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(title='Segment', framealpha=0.7)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/ltv_vs_prob_alive.png', dpi=150, bbox_inches='tight')
plt.show()

### 9c. Top 20 Customers by Predicted LTV

In [ ]:
top20 = rfm_repeat.nlargest(20, 'predicted_ltv_12m')[[
    'frequency', 'recency', 'T', 'monetary_value', 'predicted_ltv_12m', 'prob_alive'
]].copy()

fig, ax = plt.subplots(figsize=(11, 6))

colors = [CHOCOLATE if p > 0.7 else CAMEL for p in top20['prob_alive']]
bars = ax.barh(
    range(len(top20)),
    top20['predicted_ltv_12m'].values,
    color=colors,
    edgecolor='white',
    linewidth=0.4
)

ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f'Customer {cid}' for cid in top20.index], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Predicted 12-Month LTV (£)')
ax.set_title('Top 20 Customers by Predicted LTV')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.grid(axis='x', alpha=0.4)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=CHOCOLATE, label='Prob. alive > 70%'),
    Patch(facecolor=CAMEL,     label='Prob. alive ≤ 70%')
]
ax.legend(handles=legend_elements, framealpha=0.7, fontsize=8)

plt.tight_layout()
plt.savefig('outputs/top20_customers.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Export Results

In [ ]:
output = rfm_repeat[[
    'frequency', 'recency', 'T', 'monetary_value',
    'prob_alive', 'predicted_ltv_12m', 'ltv_segment'
]].sort_values('predicted_ltv_12m', ascending=False)

output.to_csv('outputs/ltv_results.csv')
print(f'Exported {len(output):,} customer LTV scores → outputs/ltv_results.csv ✅')
output.head(10)

---
## 11. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Champions (~25% of customers) drive the majority of predicted revenue** — a classic power-law distribution |
| 2 | **A subset of high-LTV customers have low probability of being alive** — they are at-risk and require urgent re-engagement |
| 3 | **Average order value is a stronger LTV driver than purchase frequency** for the top segment |
| 4 | **Low-segment customers are largely dormant** — broad re-engagement campaigns targeting them are likely inefficient |

---

### 💡 Recommendations

**1. Protect your Champions**  
The top-quartile customers generate a disproportionate share of revenue. A dedicated VIP programme (early access, loyalty rewards, account management) could significantly reduce churn in this segment.

**2. Reactivate high-LTV, low-probability-alive customers**  
The top-left quadrant of the scatter plot (high LTV, low prob. alive) is the highest-ROI re-engagement opportunity. A personalised win-back campaign — referencing their past purchase history — is warranted here.

**3. Upsell to loyal mid-tier customers**  
High-frequency customers in the Mid and High segments have strong retention but lower order values. Targeted bundles and cross-sell recommendations could increase AOV and move them into the Champions tier.

**4. Suppress low-segment customers from expensive campaigns**  
Low-segment customers with low probability of being alive should be excluded from paid re-engagement. A low-cost email nudge (or no contact at all) is the appropriate strategy.

**5. Use LTV scores for paid media bidding**  
Export the LTV scores to your CRM and feed them into Google/Meta audience targeting — bid higher for lookalikes of your Champions, lower for lookalikes of Low-segment customers.

---

*Analysis by Danai Avratoglou | Dataset: UCI Online Retail | Model: BG/NBD + Gamma-Gamma (lifetimes library)*